In [2]:
import pandas as pd
import requests
import json

# Weather data source: NOAA National Centers for Environmental Information (NCEI).

TOKEN = "fmMZOwfoaPDSXZNdwClrfpATwtcoFXOs"

headers = {
    "token": TOKEN
}

## Code to find stations

In [97]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

params = {
    "datasetid": "GHCND",
    "datatypeid": "TMAX",
    "locationid": "FIPS:55",   # Wisconsin
    "startdate": "2025-10-01",
    "enddate": "2026-06-30",
    "limit": 1000
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

stations = response.json()["results"]

for station in stations:
    print(
        station["id"]," - ",
        station["name"]," - ",
        station["mindate"]," - ",
        station["maxdate"]
    )

GHCND:USC00470045  -  AFTON JANESVILLE WWTP, WI US  -  1987-08-01  -  2026-07-06
GHCND:USC00470124  -  ALMA DAM 4, WI US  -  1936-11-01  -  2026-07-06
GHCND:USC00470265  -  APPLETON, WI US  -  1893-01-01  -  2026-07-06
GHCND:USC00470273  -  UW ARBORETUM MADISON, WI US  -  1971-10-01  -  2026-07-01
GHCND:USC00470308  -  ARLINGTON, WI US  -  1962-06-01  -  2026-07-06
GHCND:USC00470347  -  ASHLAND 3 S, WI US  -  1998-07-01  -  2026-07-06
GHCND:USC00470382  -  AUGUSTA RANGER STATION, WI US  -  2003-03-01  -  2026-07-06
GHCND:USC00470516  -  BARABOO WWTP, WI US  -  1893-01-01  -  2026-07-06
GHCND:USC00470604  -  BAYFIELD FISH HATCHERY, WI US  -  2015-10-01  -  2026-07-06
GHCND:USC00470645  -  BEAVER DAM WWTP, WI US  -  1893-01-01  -  2026-07-06
GHCND:USC00470652  -  BELGIUM WWTP, WI US  -  2009-10-01  -  2026-07-06
GHCND:USC00470696  -  BELOIT, WI US  -  1893-01-01  -  2026-07-06
GHCND:USC00470742  -  BERLIN WWTP, WI US  -  2004-11-01  -  2026-07-06
GHCND:USC00470773  -  BIG FALLS HYDRO, WI

- USC (U.S. Cooperative Observer - Volunteer observer (COOP)) - common data reported: Temperature, precipitation, snowfall, snow depth, some weather types
- USW (U.S. Weather Bureau / FAA / National Weather Service - Airports and official weather offices) - common data reported: Temperature, precipitation, wind, pressure, humidity, visibility, hourly observations
- USR (U.S. Reference Network (or special federal reference stations in NOAA IDs) - Specialized automated stations) - common data reported: High-quality climate observations, often automated

# Potential Canadates
- USC00473058 - GERMANTOWN WASTEWATER UTILITY
- USC00478937 - WAUKESHA WWTP
- USC00471062 - BROOKFIELD WWTP ***** great for temp data and precip but no weather activity data
- USW00014839 - MILWAUKEE MITCHELL AIRPORT **** great to get other data like fog and weather activity

### The Brookfield WWTP Station for our basic weather activity and temperature

In [66]:
import requests

url = "https://www.ncei.noaa.gov/access/services/search/v1/data"

params = {
    "dataset": "daily-summaries",
    "stations": "USC00471062",
    "startDate": "2025-10-01",
    "endDate": "2026-06-30",
    "limit": 100
}

response = requests.get(url, params=params)

print(response.url)
print(response.status_code)
print(response.text[:2000])

response.raise_for_status()

data = response.json()
print(json.dumps(data, indent=4))

https://www.ncei.noaa.gov/access/services/search/v1/data?dataset=daily-summaries&stations=USC00471062&startDate=2025-10-01&endDate=2026-06-30&limit=100
200
{"dataTypes":{"docCountError":0,"buckets":[{"docCount":1,"key":"PRCP"},{"docCount":1,"key":"SNOW"},{"docCount":1,"key":"SNWD"},{"docCount":1,"key":"TMAX"},{"docCount":1,"key":"TMIN"},{"docCount":1,"key":"TOBS"},{"docCount":1,"key":"WT01"},{"docCount":1,"key":"WT03"},{"docCount":1,"key":"WT04"},{"docCount":1,"key":"WT11"}],"sumOfOtherDocCounts":0},"endDate":"2026-07-04T23:59:59","count":1,"bounds":{"bottomRight":null,"topLeft":null},"totalFileSize":1324089.0,"stations":{"docCountError":0,"buckets":[{"docCount":1,"key":"USC00471062"}],"sumOfOtherDocCounts":0},"totalCount":132438,"results":[{"tar":"daily-summaries-latest.tar.gz","endDate":"2026-07-04T23:59:59","boundingPoints":[{"coordinates":[-88.17746,43.05209],"type":"point"}],"filePath":"/data/daily-summaries/access/USC00471062.csv","stations":[{"dataTypes":[{"coverage":100.0,"endD

### The Brookfield WWTP Station the best one yet!!
- the only caveat is that it has the WT weather columns but this is a USC station and they never report those types of weather activity so there is nothing in these data types

In [89]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes"

params = {
    "datasetid": "GHCND",
    "stationid": "GHCND:USC00471062",
    "limit": 1000
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

for item in response.json()["results"]:
    print(item["id"], "-", item["name"])

PRCP - Precipitation
SNOW - Snowfall
SNWD - Snow depth
TMAX - Maximum temperature
TMIN - Minimum temperature
TOBS - Temperature at the time of observation
WT01 - Fog, ice fog, or freezing fog (may include heavy fog)
WT03 - Thunder
WT04 - Ice pellets, sleet, snow pellets, or small hail" 
WT11 - High or damaging winds


In [8]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"

rows = []

params = [
    ("datasetid", "GHCND"),
    ("stationid", "GHCND:USC00471062"),   # Brookfield
    ("startdate", "2025-10-01"),
    ("enddate", "2026-06-30"),
    ("limit", 1000),
    ("offset", 1)
]

response = requests.get(url, headers=headers, params=params)

print(response.url)

response.raise_for_status()

data = response.json()
rows.extend(data.get("results", []))

df = pd.DataFrame(rows)

df["date"] = pd.to_datetime(df["date"]).dt.date

weather_df = df.pivot_table(
    index=["date", "station"],
    columns="datatype",
    values="value",
    aggfunc="first"
).reset_index()

# print(df.head())
# print(df.shape)
weather_df

https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid=GHCND%3AUSC00471062&startdate=2025-10-01&enddate=2026-06-30&limit=1000&offset=1


datatype,date,station,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS
0,2025-10-01,GHCND:USC00471062,0.0,0.0,0.0,261.0,133.0,133.0
1,2025-10-02,GHCND:USC00471062,0.0,0.0,0.0,233.0,128.0,128.0
2,2025-10-03,GHCND:USC00471062,0.0,0.0,0.0,278.0,122.0,133.0
3,2025-10-04,GHCND:USC00471062,0.0,0.0,0.0,300.0,133.0,150.0
4,2025-10-05,GHCND:USC00471062,0.0,0.0,0.0,300.0,150.0,178.0
...,...,...,...,...,...,...,...,...
162,2026-03-12,GHCND:USC00471062,15.0,0.0,0.0,17.0,-22.0,-11.0
163,2026-03-13,GHCND:USC00471062,15.0,0.0,0.0,67.0,-11.0,33.0
164,2026-03-14,GHCND:USC00471062,0.0,0.0,0.0,44.0,-61.0,-61.0
165,2026-03-15,GHCND:USC00471062,8.0,0.0,0.0,22.0,-61.0,0.0


In [17]:
url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "daily-summaries",
    "stations": "USC00471062",   # Milwaukee Mitchell Airport
    "startDate": "2025-10-01",
    "endDate": "2026-06-30",
    "format": "json",
    "units": "standard",
    "includeAttributes": "false"
}
response = requests.get(url, params=params)
data = response.json()

mill_df = pd.DataFrame(data)

mill_df

,DATE,STATION,SNOW,TMAX,TMIN,PRCP,TOBS,SNWD
0,2025-10-01,USC00471062,0.0,79,56,0.00,56,0.0
1,2025-10-02,USC00471062,0.0,74,55,0.00,55,0.0
2,2025-10-03,USC00471062,0.0,82,54,0.00,56,0.0
3,2025-10-04,USC00471062,0.0,86,56,0.00,59,0.0
4,2025-10-05,USC00471062,0.0,86,59,0.00,64,0.0
...,...,...,...,...,...,...,...,...
268,2026-06-26,USC00471062,NaN,76,52,0.00,62,NaN
269,2026-06-27,USC00471062,NaN,69,53,0.00,58,NaN
270,2026-06-28,USC00471062,NaN,74,53,0.00,64,NaN
271,2026-06-29,USC00471062,NaN,80,64,0.00,76,NaN


### The Milwaukee Mitchell Airport dataset - Great for our more detailed dataset!!

In [61]:
url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "daily-summaries",
    "stations": "USW00014839",   # Milwaukee Mitchell Airport
    "startDate": "2025-10-01",
    "endDate": "2026-06-30",
    "format": "json",
    "units": "standard",
    "includeAttributes": "false"
}

response = requests.get(url, params=params)
print(response.url)
print(response.status_code)
print(response.text[:1000])

https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00014839&startDate=2025-10-01&endDate=2026-06-30&format=json&units=standard&includeAttributes=false
200
[
{"WSF2":"16.1","DATE":"2025-10-01","WDF2":"  110","AWND":"9.62","STATION":"USW00014839","WSF5":"19.9","WDF5":"  110","SNOW":"0.0","TMAX":"71","TMIN":"62","PRCP":"0.00","SNWD":"0.0"}
,{"WSF2":"16.1","DATE":"2025-10-02","WDF2":"  130","AWND":"7.38","STATION":"USW00014839","WSF5":"19.0","WDF5":"  140","SNOW":"0.0","TMAX":"74","TMIN":"62","PRCP":"0.00","SNWD":"0.0"}
,{"WSF2":"14.1","DATE":"2025-10-03","WDF2":"  120","AWND":"6.26","STATION":"USW00014839","WSF5":"16.1","WDF5":"  120","SNOW":"0.0","TMAX":"87","TMIN":"63","PRCP":"0.00","SNWD":"0.0"}
,{"WSF2":"19.9","DATE":"2025-10-04","WDF2":"  220","AWND":"9.40","STATION":"USW00014839","WSF5":"28.0","WDF5":"  220","SNOW":"0.0","TMAX":"87","TMIN":"66","PRCP":"0.00","SNWD":"0.0"}
,{"WSF2":"23.9","DATE":"2025-10-05","WDF2":"  200","AWND":"12.30","STATION":"

### This is the columns this station returns
- The only way for there to be a column to be returned into a data frame there's got to be data in it

In [16]:
url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "daily-summaries",
    "stations": "USW00014839",   # Milwaukee Mitchell Airport
    "startDate": "2025-10-01",
    "endDate": "2026-06-30",
    "format": "json",
    "units": "standard",
    "includeAttributes": "false"
}
response = requests.get(url, params=params)
data = response.json()

mill_df = pd.DataFrame(data)

mill_df

,WSF2,DATE,WDF2,AWND,STATION,WSF5,WDF5,SNOW,TMAX,TMIN,PRCP,SNWD,WT01,WT03,WT08,WT02,WT09,WT06,WT04,WT05
0,16.1,2025-10-01,110,9.62,USW00014839,19.9,110,0.0,71,62,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16.1,2025-10-02,130,7.38,USW00014839,19.0,140,0.0,74,62,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14.1,2025-10-03,120,6.26,USW00014839,16.1,120,0.0,87,63,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19.9,2025-10-04,220,9.40,USW00014839,28.0,220,0.0,87,66,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,23.9,2025-10-05,200,12.30,USW00014839,34.0,220,0.0,82,64,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,12.1,2026-06-26,40,4.92,USW00014839,15.0,20,0.0,63,54,0.00,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
269,14.1,2026-06-27,30,6.26,USW00014839,17.9,50,0.0,71,57,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
270,19.9,2026-06-28,130,6.71,USW00014839,27.1,140,0.0,74,62,0.01,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,23.9,2026-06-29,200,12.53,USW00014839,32.0,200,0.0,92,67,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Now we have a lot more data to work with this station!!


In [98]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/datatypes"

params = {
    "datasetid": "GHCND",
    "stationid": "GHCND:USW00014839",
    "limit": 1000
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

for item in response.json()["results"]:
    print(item["id"], "-", item["name"])

ACMH - Average cloudiness midnight to midnight from manual observations
ACSH - Average cloudiness sunrise to sunset from manual observations
AWND - Average wind speed
FMTM - Time of fastest mile or fastest 1-minute wind
FRGB - Base of frozen ground layer
FRGT - Top of frozen ground layer
FRTH - Thickness of frozen ground layer
GAHT - Difference between river and gauge height
PGTM - Peak gust time
PRCP - Precipitation
PSUN - Daily percent of possible sunshine for the period
SNOW - Snowfall
SNWD - Snow depth
TAVG - Average Temperature.
THIC - Thickness of ice on water
TMAX - Maximum temperature
TMIN - Minimum temperature
TOBS - Temperature at the time of observation
TSUN - Total sunshine for the period
WDF1 - Direction of fastest 1-minute wind
WDF2 - Direction of fastest 2-minute wind
WDF5 - Direction of fastest 5-second wind
WDFG - Direction of peak wind gust
WDFM - Fastest mile wind direction
WESD - Water equivalent of snow on the ground
WSF1 - Fastest 1-minute wind speed
WSF2 - Fastes

### Looked into LCD (Local Climatological Data) for more detailed data, but the data are a little old (about a year old data)

In [54]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations"

params = {
    "datasetid": "LCD",
    "locationid": "FIPS:55",   # Wisconsin
    "startdate": "2020-10-01",
    "enddate": "2025-06-30",
    "limit": 1000
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

data = response.json()

print(json.dumps(data, indent=4, sort_keys=True))

{
    "metadata": {
        "resultset": {
            "count": 64,
            "limit": 1000,
            "offset": 1
        }
    },
    "results": [
        {
            "datacoverage": 1,
            "elevation": 262.1,
            "elevationUnit": "METERS",
            "id": "WBAN:00132",
            "latitude": 44.033,
            "longitude": -89.3,
            "maxdate": "2025-08-25",
            "mindate": "2014-07-31",
            "name": "WAUTOMA MUNICIPAL AIRPORT, WI US"
        },
        {
            "datacoverage": 1,
            "elevation": 312.1,
            "elevationUnit": "METERS",
            "id": "WBAN:00183",
            "latitude": 42.683,
            "longitude": -90.45,
            "maxdate": "2025-08-25",
            "mindate": "2009-01-01",
            "name": "PLATTEVILLE MUNICIPAL AIRPORT, WI US"
        },
        {
            "datacoverage": 1,
            "elevation": 248.1,
            "elevationUnit": "METERS",
            "id": "WBAN:00185",
  

In [44]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/stations/GHCND:USC00471062"

response = requests.get(url, headers=headers)
response.raise_for_status()

station = response.json()
print(station)

{'elevation': 253, 'mindate': '2007-01-01', 'maxdate': '2026-07-04', 'latitude': 43.05209, 'name': 'BROOKFIELD WWTP, WI US', 'datacoverage': 1, 'id': 'GHCND:USC00471062', 'elevationUnit': 'METERS', 'longitude': -88.17746}


In [ ]:
from io import StringIO
import pandas as pd
import requests

url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "local-climatological-data",
    "stations": "WBAN:04897",
    "startDate": "2025-10-01",
    "endDate": "2025-10-31",
    "format": "csv"
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.text[:500])

df = pd.read_csv(StringIO(response.text))

print(df.head())
print(df.columns)

200
"STATION","DATE","REPORT_TYPE","SOURCE","AWND","BackupDirection","BackupDistance","BackupDistanceUnit","BackupElements","BackupElevation","BackupElevationUnit","BackupEquipment","BackupLatitude","BackupLongitude","BackupName","CDSD","CLDD","DSNW","DYHF","DYTS","DailyAverageDewPointTemperature","DailyAverageDryBulbTemperature","DailyAverageRelativeHumidity","DailyAverageSeaLevelPressure","DailyAverageStationPressure","DailyAverageWetBulbTemperature","DailyAverageWindSpeed","DailyCoolingDegreeDays
Empty DataFrame
Columns: [STATION, DATE, REPORT_TYPE, SOURCE, AWND, BackupDirection, BackupDistance, BackupDistanceUnit, BackupElements, BackupElevation, BackupElevationUnit, BackupEquipment, BackupLatitude, BackupLongitude, BackupName, CDSD, CLDD, DSNW, DYHF, DYTS, DailyAverageDewPointTemperature, DailyAverageDryBulbTemperature, DailyAverageRelativeHumidity, DailyAverageSeaLevelPressure, DailyAverageStationPressure, DailyAverageWetBulbTemperature, DailyAverageWindSpeed, DailyCoolingDegreeD

## Finished Power BI Code

### GHCND Data Option - WAY TO SLOW!!!

In [87]:
import pandas as pd
import requests

TOKEN = "fmMZOwfoaPDSXZNdwClrfpATwtcoFXOs"

headers = {
    "token": TOKEN
}

url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"

rows = []
offset = 1
limit = 1000

while True:
    params = [
        ("datasetid", "GHCND"),
        ("stationid", "GHCND:USC00471062"),   # Brookfield
        ("stationid", "GHCND:USW00014839"),   # Milwaukee Mitchell
        ("startdate", "2025-10-01"),
        ("enddate", "2026-06-30"),
        ("limit", limit),
        ("offset", offset)
    ]

    response = requests.get(url, headers=headers, params=params)

    print(response.url)  # helpful for debugging

    response.raise_for_status()

    data = response.json()
    rows.extend(data.get("results", []))

    metadata = data.get("metadata", {}).get("resultset", {})
    count = metadata.get("count", 0)

    if offset + limit > count:
        break

    offset += limit

df = pd.DataFrame(rows)

df["date"] = pd.to_datetime(df["date"]).dt.date

weather_df = df.pivot_table(
    index=["date", "station"],
    columns="datatype",
    values="value",
    aggfunc="first"
).reset_index()

# print(df.head())
# print(df.shape)
weather_df

https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid=GHCND%3AUSC00471062&stationid=GHCND%3AUSW00014839&startdate=2025-10-01&enddate=2026-06-30&limit=1000&offset=1
https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid=GHCND%3AUSC00471062&stationid=GHCND%3AUSW00014839&startdate=2025-10-01&enddate=2026-06-30&limit=1000&offset=1001
https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid=GHCND%3AUSC00471062&stationid=GHCND%3AUSW00014839&startdate=2025-10-01&enddate=2026-06-30&limit=1000&offset=2001
https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid=GHCND%3AUSC00471062&stationid=GHCND%3AUSW00014839&startdate=2025-10-01&enddate=2026-06-30&limit=1000&offset=3001
https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&stationid=GHCND%3AUSC00471062&stationid=GHCND%3AUSW00014839&startdate=2025-10-01&enddate=2026-06-30&limit=1000&offset=4001


datatype,date,station,AWND,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WDF2,...,WSF2,WSF5,WT01,WT02,WT03,WT04,WT05,WT06,WT08,WT09
0,2025-10-01,GHCND:USC00471062,NaN,0.0,0.0,0.0,261.0,133.0,133.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-01,GHCND:USW00014839,43.0,0.0,0.0,0.0,217.0,167.0,NaN,110.0,...,72.0,89.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-02,GHCND:USC00471062,NaN,0.0,0.0,0.0,233.0,128.0,128.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-02,GHCND:USW00014839,33.0,0.0,0.0,0.0,233.0,167.0,NaN,130.0,...,72.0,85.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-10-03,GHCND:USC00471062,NaN,0.0,0.0,0.0,278.0,122.0,133.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,2026-06-28,GHCND:USW00014839,30.0,3.0,0.0,0.0,233.0,167.0,NaN,130.0,...,89.0,121.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
542,2026-06-29,GHCND:USC00471062,NaN,0.0,NaN,NaN,267.0,178.0,244.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
543,2026-06-29,GHCND:USW00014839,56.0,0.0,0.0,0.0,333.0,194.0,NaN,200.0,...,107.0,143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
544,2026-06-30,GHCND:USC00471062,NaN,0.0,NaN,NaN,333.0,244.0,272.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"

params = [
    ("datasetid", "GHCND"),
    ("stationid", "GHCND:USC00471062"),   # Brookfield
    ("stationid", "GHCND:USW00014839"),   # Milwaukee Mitchell
    ("startdate", "2025-10-01"),
    ("enddate", "2026-06-30"),
    ("limit", 1000),
    ("offset", 1)
]

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

data = response.json()
print(data)

{'metadata': {'resultset': {'offset': 1, 'count': 4498, 'limit': 1000}}, 'results': [{'date': '2025-10-01T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:USC00471062', 'attributes': ',,7,0800', 'value': 0}, {'date': '2025-10-01T00:00:00', 'datatype': 'SNOW', 'station': 'GHCND:USC00471062', 'attributes': ',,7,', 'value': 0}, {'date': '2025-10-01T00:00:00', 'datatype': 'SNWD', 'station': 'GHCND:USC00471062', 'attributes': ',,7,0800', 'value': 0}, {'date': '2025-10-01T00:00:00', 'datatype': 'TMAX', 'station': 'GHCND:USC00471062', 'attributes': ',,7,0800', 'value': 261}, {'date': '2025-10-01T00:00:00', 'datatype': 'TMIN', 'station': 'GHCND:USC00471062', 'attributes': ',,7,0800', 'value': 133}, {'date': '2025-10-01T00:00:00', 'datatype': 'TOBS', 'station': 'GHCND:USC00471062', 'attributes': ',,7,0800', 'value': 133}, {'date': '2025-10-01T00:00:00', 'datatype': 'AWND', 'station': 'GHCND:USW00014839', 'attributes': ',,1,', 'value': 43}, {'date': '2025-10-01T00:00:00', 'datatype': 'PRCP', 'st

In [25]:
import requests
import pandas as pd

url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "daily-summaries",
    "stations": "USC00471062,USW00014839",
    "startDate": "2025-10-01",
    "endDate": "2026-06-30",
    "format": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

df = pd.DataFrame(data)

print(df.columns.tolist())

['DATE', 'STATION', 'SNOW', 'TMAX', 'TMIN', 'PRCP', 'TOBS', 'SNWD', 'WSF2', 'WDF2', 'AWND', 'WSF5', 'WDF5', 'WT01', 'WT03', 'WT08', 'WT02', 'WT09', 'WT06', 'WT04', 'WT05']


## The Daily-Summary option - Much faster!!!
- takes only a little over 2 minutes to get all the data from the stations compared to 10
- I also added caching to make the process even quicker so we are not having to grab the data everytime

In [ ]:
import os
import pandas as pd
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
from time import perf_counter
from io import StringIO

#-----------------------------------
# HubSpot Data from Ice Rink Date
#-----------------------------------

load_dotenv(r"D:\other-files\school\database_dev\Windigo Internship\HUBSPOT_KEY.env")

service_key = os.getenv("HUBSPOT_SERVICE_KEY")

if not service_key:
    raise ValueError("HUBSPOT_SERVICE_KEY was not loaded")

hubspot_url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers_hub = {
    "Authorization": f"Bearer {service_key}"
}

response_hub = requests.get(
    "https://api.hubapi.com/crm/v3/objects/contacts",
    headers=headers_hub
)

params_hub = {
    "limit": 100,
    "properties": "date"
}

rows = []

while hubspot_url:
    r = requests.get(hubspot_url, headers=headers_hub, params=params_hub)
    r.raise_for_status()
    data = r.json()

    for item in data.get("results", []):
        props = item.get("properties", {})

        row = {
            "id": item.get("id")
        }

        row["date"] = props.get("date")

        rows.append(row)

    hubspot_url = data.get("paging", {}).get("next", {}).get("link")

    params_hub = None

ice_df = pd.DataFrame(rows)

ice_df["date"] = pd.to_datetime(ice_df["date"])

start_date = ice_df['date'].min().strftime("%Y-%m-%d")
end_date = ice_df['date'].max().strftime("%Y-%m-%d")

#------------------------------------------------
# NOAA Data from Brookfield and Milwaukee Airport
#------------------------------------------------

CACHE_DIR = "weather_cache"

CACHE_PATH = os.path.join(
    CACHE_DIR,
    "southeast_wisconsin_weather.parquet"
)

os.makedirs(CACHE_DIR, exist_ok=True)

def fetch_noaa_data(stations, startDate, endDate):
    noaa_url = "https://www.ncei.noaa.gov/access/services/data/v1"

    params = {
        "dataset": "daily-summaries",
        "stations": stations,
        "startDate": startDate.strftime("%Y-%m-%d"),
        "endDate": endDate.strftime("%Y-%m-%d"),
        "dataTypes": (
            "PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,"
            "WT03,WT04,WT05,WT06,WT09"
        ),
        "format": "json",
        "units": "standard",
        "includeAttributes": "false",
        "includeStationName": "false",
        "includeStationLocation": "false"
    }
    start = perf_counter()
    response = requests.get(noaa_url, params=params, timeout=60)
    request_finished = perf_counter()
    response.raise_for_status()

    data = response.json()

    if not data:
        return pd.DataFrame()
    
    new_df = pd.DataFrame(data)

    if "DATE" in new_df.columns:
        new_df["DATE"] = pd.to_datetime(new_df["DATE"], errors="coerce").dt.normalize()
    
    parse_finished = perf_counter()

    print(
        f"NOAA request: "
        f"{request_finished - start:.2f} seconds"
    )

    print(
        f"JSON parsing: "
        f"{parse_finished - request_finished:.2f} seconds"
    )

    print(
        f"Response size: "
        f"{len(response.content) / 1024:.1f} KB"
    )

    return new_df

def get_weather_data(stations, startDate, endDate, force_refresh=False):
    startDate = pd.Timestamp(startDate).normalize()
    endDate = pd.Timestamp(endDate).normalize()

    if force_refresh or not os.path.exists(CACHE_PATH):
        
        print("Creating weather cache from NOAA")

        weather = fetch_noaa_data(
            stations=stations,
            startDate=startDate,
            endDate=endDate
        )

        if not weather.empty:
            weather.to_parquet(
                CACHE_PATH,
                index=False,
                compression="snappy"
            )
        return weather
    
    print("Reading existing weather cache")

    cached = pd.read_parquet(CACHE_PATH)
    cached["DATE"] = pd.to_datetime(cached["DATE"]).dt.normalize()

    station_list = [station.strip() for station in stations.split(",")]

    cached = cached[cached["STATION"].isin(station_list)].copy()

    pieces = [cached]

    cached_start = cached["DATE"].min()
    cached_end = cached["DATE"].max()

    # Fetch dates before the existing cache
    if startDate < cached_start:
        earlier_end = cached_start - pd.Timedelta(days=1)

        print(
            f"Fetching earlier NOAA data: "
            f"{start_date.date()} through {earlier_end.date()}"
        )

        earlier = fetch_noaa_data(
            stations=stations,
            startDate=startDate,
            endDate=earlier_end
        )

        if not earlier.empty:
            pieces.append(earlier)

    # Fetch dates after the existing cache
    if endDate > cached_end:
        later_start = cached_end + pd.Timedelta(days=1)

        print(
            f"Fetching newer NOAA data: "
            f"{later_start.date()} through {end_date.date()}"
        )

        later = fetch_noaa_data(
            stations=stations,
            startDate=later_start,
            endDate=endDate
        )
        if not later.empty:
            pieces.append(later)

    weather = pd.concat(pieces, ignore_index=True)

    weather["DATE"] = pd.to_datetime(weather["DATE"]).dt.normalize()

    weather = weather.drop_duplicates(
        subset=["STATION", "DATE"],
        keep="last"
    )

    weahter = weather.sort_values(
        ["STATION", "DATE"]
    ).reset_index(drop=True)

    weather.to_parquet(
        CACHE_PATH,
        index=False,
        compression="snappy"
    )

    # Return only the range currently required by HubSpot
    weather = weather[
        weather["DATE"].between(start_date, end_date)
    ].copy()

    return weather

df = get_weather_data(
    stations = "USC00471062,USW00014839",
    startDate = start_date,
    endDate = end_date
)

df["DATE"] = pd.to_datetime(df["DATE"])

brookfield = df[df["STATION"] == "USC00471062"].copy()
mitchell = df[df["STATION"] == "USW00014839"].copy()

brookfield_keep = [
    "DATE", "SNOW", "TMAX", "TMIN", "PRCP", "TOBS", "SNWD"
]

w_cols = [
    c for c in mitchell.columns 
    if c.startswith(("WT", "WS", "WD")) or c == "AWND"
]

mitchell_keep = ["DATE"] + w_cols

weather_df = brookfield[brookfield_keep].merge(
    mitchell[mitchell_keep],
    on="DATE",
    how="left"
)

to_int = ['TMAX', 'TMIN', 'PRCP', 'TOBS', 'WT03', 'WT09', 'WT06', 'WT04', 'SNOW']

for int in to_int:
    weather_df[int] = pd.to_numeric(weather_df[int])

weather_df = weather_df.rename(
    columns={
        'TOBS': 'Temperature Observed',
        'SNWD': 'Snow Depth',
        'WT03': 'Thunder',
        'WT04': 'Ice Pellets - Sleet - Snow Pellets - Small Hail',
        'WT05': 'Hail',
        'WT06': 'Glaze or Rime',
        'WT09': 'Blowing or Drifting Snow'
    }
)

weather_df = weather_df.fillna(0)

weather_df

Creating weather cache from NOAA
NOAA request: 127.03 seconds
JSON parsing: 0.00 seconds
Response size: 74.1 KB


,DATE,SNOW,TMAX,TMIN,PRCP,Temperature Observed,Snow Depth,Fastest Two Minute Wind Speed,Fastest Five Second Wind Speed,Direction of Fastest Two Minute Wind,Average Daily Wind Speed,Direction of Fastest Five Second Wind,Fog - Ice Fog - Freezing Fog,Thunder,Smoke or Haze,Heavy Fog or Heavy Freezing Fog,Blowing or Drifting Snow,Glaze or Rime,Ice Pellets - Sleet - Snow Pellets - Small Hail,Hail
0,2025-10-13,0.0,65,55,0.00,58,0.0,8.9,13.0,160,3.80,160.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-10-14,0.0,60,46,0.20,51,0.0,19.9,25.9,30,10.96,20.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-10-15,0.0,62,49,0.24,50,0.0,15.0,21.0,40,7.61,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-10-16,0.0,57,49,0.00,50,0.0,17.0,21.0,140,7.61,130.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-10-17,0.0,64,49,0.00,57,0.0,19.9,28.0,200,11.41,190.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,2026-06-05,0.0,87,61,0.25,62,0.0,28.0,36.9,310,9.62,310.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
236,2026-06-06,0.0,76,60,0.53,65,0.0,13.0,17.0,80,5.82,70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
237,2026-06-07,0.0,84,57,0.02,61,0.0,14.1,17.9,30,6.49,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
238,2026-06-08,0.0,79,61,0.05,65,0.0,15.0,19.0,140,8.28,140.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


- NOAA request: 126.89 seconds with set datatypes params
- NOAA request: 127.03 seconds without set datatypes params
- NOAA request: 126.74 seconds using CSV and set datatypes

In [169]:
import os
import pandas as pd
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
from time import perf_counter
from io import StringIO
import openmeteo_requests
import requests_cache
from retry_requests import retry
from functools import reduce


#-----------------------------------
# HubSpot Data from Ice Rink Date
#-----------------------------------

load_dotenv(r"D:\other-files\school\database_dev\Windigo Internship\HUBSPOT_KEY.env")

service_key = os.getenv("HUBSPOT_SERVICE_KEY")

if not service_key:
    raise ValueError("HUBSPOT_SERVICE_KEY was not loaded")

hubspot_url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers_hub = {
    "Authorization": f"Bearer {service_key}"
}

response_hub = requests.get(
    "https://api.hubapi.com/crm/v3/objects/contacts",
    headers=headers_hub
)

params_hub = {
    "limit": 100,
    "properties": "date"
}

rows = []

while hubspot_url:
    r = requests.get(hubspot_url, headers=headers_hub, params=params_hub)
    r.raise_for_status()
    data = r.json()

    for item in data.get("results", []):
        props = item.get("properties", {})

        row = {
            "id": item.get("id")
        }

        row["date"] = props.get("date")

        rows.append(row)

    hubspot_url = data.get("paging", {}).get("next", {}).get("link")

    params_hub = None

ice_df = pd.DataFrame(rows)

ice_df["date"] = pd.to_datetime(ice_df["date"])

start_date = ice_df['date'].min().strftime("%Y-%m-%d")
end_date = ice_df['date'].max().strftime("%Y-%m-%d")

#------------------------------------------------
# NOAA Data from Brookfield and Milwaukee Airport
#------------------------------------------------

CACHE_DIR = "weather_cache"

CACHE_PATH = os.path.join(
    CACHE_DIR,
    "southeast_wisconsin_weather.parquet"
)

os.makedirs(CACHE_DIR, exist_ok=True)

def fetch_noaa_data(stations, startDate, endDate):
    noaa_url = "https://www.ncei.noaa.gov/access/services/data/v1"

    params = {
        "dataset": "daily-summaries",
        "stations": stations,
        "startDate": startDate.strftime("%Y-%m-%d"),
        "endDate": endDate.strftime("%Y-%m-%d"),
        "dataTypes": (
            "PRCP,SNOW,SNWD,TMAX,TMIN,"
            "WT03,WT04,WT05,WT06,WT09"
        ),
        "format": "csv",
        "units": "standard",
        "includeAttributes": "false",
        "includeStationName": "false",
        "includeStationLocation": "false"
    }
    start = perf_counter()
    response = requests.get(noaa_url, params=params, timeout=60)
    request_finished = perf_counter()
    response.raise_for_status()

    if not data:
        return pd.DataFrame()

    new_df = pd.read_csv(StringIO(response.text))

    if "DATE" in new_df.columns:
        new_df["DATE"] = pd.to_datetime(new_df["DATE"], errors="coerce").dt.normalize()
    
    parse_finished = perf_counter()

    print(
        f"NOAA request: "
        f"{request_finished - start:.2f} seconds"
    )

    print(
        f"CSV parsing: "
        f"{parse_finished - request_finished:.2f} seconds"
    )

    print(
        f"Response size: "
        f"{len(response.content) / 1024:.1f} KB"
    )

    return new_df

def get_weather_data(stations, startDate, endDate, force_refresh=False):
    startDate = pd.Timestamp(startDate).normalize()
    endDate = pd.Timestamp(endDate).normalize()

    if force_refresh or not os.path.exists(CACHE_PATH):
        
        print("Creating weather cache from NOAA")

        weather = fetch_noaa_data(
            stations=stations,
            startDate=startDate,
            endDate=endDate
        )

        if not weather.empty:
            weather.to_parquet(
                CACHE_PATH,
                index=False,
                compression="snappy"
            )
        return weather
    
    print("Reading existing weather cache")

    cached = pd.read_parquet(CACHE_PATH)
    cached["DATE"] = pd.to_datetime(cached["DATE"]).dt.normalize()

    station_list = [station.strip() for station in stations.split(",")]

    cached = cached[cached["STATION"].isin(station_list)].copy()

    pieces = [cached]

    cached_start = cached["DATE"].min()
    cached_end = cached["DATE"].max()

    # Fetch dates before the existing cache
    if startDate < cached_start:
        earlier_end = cached_start - pd.Timedelta(days=1)

        print(
            f"Fetching earlier NOAA data: "
            f"{start_date.date()} through {earlier_end.date()}"
        )

        earlier = fetch_noaa_data(
            stations=stations,
            startDate=startDate,
            endDate=earlier_end
        )

        if not earlier.empty:
            pieces.append(earlier)

    # Fetch dates after the existing cache
    if endDate > cached_end:
        later_start = cached_end + pd.Timedelta(days=1)

        print(
            f"Fetching newer NOAA data: "
            f"{later_start.date()} through {end_date.date()}"
        )

        later = fetch_noaa_data(
            stations=stations,
            startDate=later_start,
            endDate=endDate
        )
        if not later.empty:
            pieces.append(later)

    weather = pd.concat(pieces, ignore_index=True)

    weather["DATE"] = pd.to_datetime(weather["DATE"]).dt.normalize()

    weather = weather.drop_duplicates(
        subset=["STATION", "DATE"],
        keep="last"
    )

    weahter = weather.sort_values(
        ["STATION", "DATE"]
    ).reset_index(drop=True)

    weather.to_parquet(
        CACHE_PATH,
        index=False,
        compression="snappy"
    )

    # Return only the range currently required by HubSpot
    weather = weather[
        weather["DATE"].between(start_date, end_date)
    ].copy()

    return weather

df = get_weather_data(
    stations = "USC00471062,USW00014839",
    startDate = start_date,
    endDate = end_date
)

#-----------------------------------
# Metro weather API
#-----------------------------------

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 43.07036380872471, 
	"longitude": -88.12443039077134,
	"start_date": start_date,
	"end_date": end_date,
	"daily": ["temperature_2m_max", "temperature_2m_min", "relative_humidity_2m_max", "relative_humidity_2m_mean", "dew_point_2m_mean", "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant"],
	"temperature_unit": "fahrenheit",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")


# Process daily data
daily = response.Daily()
daily_rh_max = daily.Variables(2).ValuesAsNumpy()
daily_rh_mean = daily.Variables(3).ValuesAsNumpy()
daily_dew_mean = daily.Variables(4).ValuesAsNumpy()
daily_wind_speed_max = daily.Variables(5).ValuesAsNumpy()
daily_wind_gusts_max = daily.Variables(6).ValuesAsNumpy()
daily_wind_dir = daily.Variables(7).ValuesAsNumpy()

# Create your dataframe
daily_data = {
    "DATE": pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    ),
    "Rel Humidity Max": daily_rh_max,
    "Rel Humidity Mean": daily_rh_mean,
    "Dew Point Mean": daily_dew_mean,
    "Wind Speed Max": daily_wind_speed_max,
    "Wind Gusts Max": daily_wind_gusts_max,
    "Wind Direction": daily_wind_dir
}

# 1. Create the DataFrame
metro_dataframe = pd.DataFrame(data=daily_data)

df["DATE"] = pd.to_datetime(df["DATE"])

brookfield = df[df["STATION"] == "USC00471062"].copy()
mitchell = df[df["STATION"] == "USW00014839"].copy()

brookfield_keep = [
    "DATE", "SNOW", "TMAX", "TMIN", "PRCP", "TOBS", "SNWD"
]

brookfield_select = brookfield[brookfield_keep].copy()

w_cols = [
    c for c in mitchell.columns 
    if c.startswith(("WT", "WS", "WD")) or c == "AWND"
]

mitchell_keep = ["DATE"] + w_cols

mitchell_select = mitchell[mitchell_keep].copy()

data_frames = [brookfield_select, mitchell_select, metro_dataframe]

weather_df = reduce(
    lambda left, right: pd.merge(left, right, on="DATE", how="left"), data_frames
)

to_int = ['TMAX', 'TMIN', 'PRCP', 'WT03', 'WT09', 'WT06', 'WT04', 'SNOW']

for int in to_int:
    weather_df[int] = pd.to_numeric(weather_df[int])

weather_df = weather_df.rename(
    columns={
        'SNWD': 'Snow Depth',
        'WT03': 'Milwaukee - Thunder',
        'WT04': 'Milwaukee - Ice Pellets / Sleet',
        'WT05': 'Milwaukee - Hail',
        'WT06': 'Milwaukee - Glaze or Rime',
        'WT09': 'Milwaukee - Blowing or Drifting Snow'
    }
)

weather_df = weather_df.fillna(0)

weather_df

Reading existing weather cache
Coordinates: 43.08087158203125°N -88.13955688476562°E
Elevation: 256.0 m asl
Timezone difference to GMT+0: 0s


ValueError: You are trying to merge on datetime64[us] and datetime64[s, UTC] columns for key 'DATE'. If you wish to proceed you should use pd.concat

In [ ]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv

#-----------------------------------
# HubSpot Data from Ice Rink Date
#-----------------------------------

load_dotenv(r"D:\other-files\school\database_dev\Windigo Internship\HUBSPOT_KEY.env")

service_key = os.getenv("HUBSPOT_SERVICE_KEY")

if not service_key:
    raise ValueError("HUBSPOT_SERVICE_KEY was not loaded")

url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers_hub = {
    "Authorization": f"Bearer {service_key}"
}

response_hub = requests.get(
    "https://api.hubapi.com/crm/v3/objects/contacts",
    headers=headers_hub
)

params_hub = {
    "limit": 100,
    "properties": "date"
}

rows = []

while url:
    r = requests.get(url, headers=headers_hub, params=params_hub)
    r.raise_for_status()
    data = r.json()

    for item in data.get("results", []):
        props = item.get("properties", {})

        row = {
            "id": item.get("id")
        }

        row["date"] = props.get("date")

        rows.append(row)

    url = data.get("paging", {}).get("next", {}).get("link")
    params = None

ice_df = pd.DataFrame(rows)

ice_df["date"] = pd.to_datetime(ice_df["date"])

start_date = ice_df['date'].min().strftime("%Y-%m-%d")
end_date = ice_df['date'].max().strftime("%Y-%m-%d")

#------------------------------------------------
# NOAA Data from Brookfield and Milwaukee Airport
#------------------------------------------------

url = "https://www.ncei.noaa.gov/access/services/data/v1"

params = {
    "dataset": "daily-summaries",
    "stations": "USC00471062,USW00014839",
    "startDate": start_date,
    "endDate": end_date,
    "format": "json",
    "units": "standard",
    "includeAttributes": "false"
}

response = requests.get(url, params=params)
response.raise_for_status()

df = pd.DataFrame(response.json())

df["DATE"] = pd.to_datetime(df["DATE"])

brookfield = df[df["STATION"] == "USC00471062"].copy()
mitchell = df[df["STATION"] == "USW00014839"].copy()

brookfield_keep = [
    "DATE", "SNOW", "TMAX", "TMIN", "PRCP", "TOBS", "SNWD"
]

w_cols = [
    c for c in mitchell.columns 
    if c.startswith(("WT", "WS", "WD")) or c == "AWND"
]

mitchell_keep = ["DATE"] + w_cols

weather_df = brookfield[brookfield_keep].merge(
    mitchell[mitchell_keep],
    on="DATE",
    how="left"
)

to_int = ['TMAX', 'TMIN', 'PRCP', 'TOBS', 'SNWD', 'WSF2', 'WDF2', 'AWND', 'WSF5', 'WDF5', 'WT01', 'WT03', 'WT08', 'WT02', 'WT09', 'WT06', 'WT04', 'WT05', 'SNOW']

for int in to_int:
    weather_df[int] = pd.to_numeric(weather_df[int])

weather_df = weather_df.rename(
    columns={
        'TOBS': 'Temperature Observed',
        'SNWD': 'Snow Depth',
        'WSF2': 'Fastest Two Minute Wind Speed',
        'WDF2': 'Direction of Fastest Two Minute Wind',
        'AWND': 'Average Daily Wind Speed',
        'WSF5': 'Fastest Five Second Wind Speed',
        'WDF5': 'Direction of Fastest Five Second Wind',
        'WT01': 'Fog - Ice Fog - Freezing Fog',
        'WT02': 'Heavy Fog or Heavy Freezing Fog',
        'WT03': 'Thunder',
        'WT04': 'Ice Pellets - Sleet - Snow Pellets - Small Hail',
        'WT05': 'Hail',
        'WT06': 'Glaze or Rime',
        'WT08': 'Smoke or Haze',
        'WT09': 'Blowing or Drifting Snow'
    }
)

weather_df

,DATE,SNOW,TMAX,TMIN,PRCP,Temperature Observed,Snow Depth,Fastest Two Minute Wind Speed,Fastest Five Second Wind Speed,Direction of Fastest Two Minute Wind,Average Daily Wind Speed,Direction of Fastest Five Second Wind,Fog - Ice Fog - Freezing Fog,Thunder,Smoke or Haze,Heavy Fog or Heavy Freezing Fog,Blowing or Drifting Snow,Glaze or Rime,Ice Pellets - Sleet - Snow Pellets - Small Hail,Hail
0,2025-10-13,0.0,65,55,0.00,58,0.0,8.9,13.0,160,3.80,160.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-14,0.0,60,46,0.20,51,0.0,19.9,25.9,30,10.96,20.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-15,0.0,62,49,0.24,50,0.0,15.0,21.0,40,7.61,50.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-16,0.0,57,49,0.00,50,0.0,17.0,21.0,140,7.61,130.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-10-17,0.0,64,49,0.00,57,0.0,19.9,28.0,200,11.41,190.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,2026-06-05,0.0,87,61,0.25,62,0.0,28.0,36.9,310,9.62,310.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN
236,2026-06-06,0.0,76,60,0.53,65,0.0,13.0,17.0,80,5.82,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
237,2026-06-07,0.0,84,57,0.02,61,0.0,14.1,17.9,30,6.49,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
238,2026-06-08,NaN,79,61,0.05,65,NaN,15.0,19.0,140,8.28,140.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


"Wind observations were obtained from Milwaukee Mitchell International Airport due to the absence of wind measurements at the Brookfield cooperative station. Mitchell Airport is approximately 10–15 miles east of the study site and experiences the same regional weather systems, making it a suitable proxy for daily wind conditions."

#### This shows what is being filled and what is not (looking like everything is being filled)
- This also shows how often a certain event occures in this dataset

In [103]:
summary = pd.DataFrame({
    "Non-Null": weather_df.notna().sum(),
    "Missing": weather_df.isna().sum(),
    "Percent Filled": (weather_df.notna().mean()*100).round(1)
})

summary.sort_values("Percent Filled", ascending=False)

,Non-Null,Missing,Percent Filled
DATE,273,0,100.0
TMAX,273,0,100.0
TMIN,273,0,100.0
TOBS,273,0,100.0
AWND,273,0,100.0
WSF2,273,0,100.0
WDF2,273,0,100.0
WSF5,272,1,99.6
WDF5,272,1,99.6
PRCP,272,1,99.6


### Lucas Open-Metro Code w/ modifications

In [167]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from dotenv import load_dotenv

#-----------------------------------
# HubSpot Data from Ice Rink Date
#-----------------------------------

load_dotenv(r"D:\other-files\school\database_dev\Windigo Internship\HUBSPOT_KEY.env")

service_key = os.getenv("HUBSPOT_SERVICE_KEY")

if not service_key:
    raise ValueError("HUBSPOT_SERVICE_KEY was not loaded")

url = "https://api.hubapi.com/crm/v3/objects/contacts"

headers_hub = {
    "Authorization": f"Bearer {service_key}"
}

response_hub = requests.get(
    "https://api.hubapi.com/crm/v3/objects/contacts",
    headers=headers_hub
)

params_hub = {
    "limit": 100,
    "properties": "date"
}

rows = []

while url:
    r = requests.get(url, headers=headers_hub, params=params_hub)
    r.raise_for_status()
    data = r.json()

    for item in data.get("results", []):
        props = item.get("properties", {})

        row = {
            "id": item.get("id")
        }

        row["date"] = props.get("date")

        rows.append(row)

    url = data.get("paging", {}).get("next", {}).get("link")
    params = None

ice_df = pd.DataFrame(rows)

ice_df["date"] = pd.to_datetime(ice_df["date"])

start_date = ice_df['date'].min().strftime("%Y-%m-%d")
end_date = ice_df['date'].max().strftime("%Y-%m-%d")

#-----------------------------------
# Metro weather API
#-----------------------------------

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 43.07036380872471, 
	"longitude": -88.12443039077134,
	"start_date": start_date,
	"end_date": end_date,
	"daily": ["temperature_2m_max", "temperature_2m_min", "relative_humidity_2m_max", "relative_humidity_2m_mean", "dew_point_2m_mean", "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant"],
	"temperature_unit": "fahrenheit",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")


# Process daily data
daily = response.Daily()
daily_max = daily.Variables(0).ValuesAsNumpy()
daily_min = daily.Variables(1).ValuesAsNumpy()
daily_rh_max = daily.Variables(2).ValuesAsNumpy()
daily_rh_mean = daily.Variables(3).ValuesAsNumpy()
daily_dew_mean = daily.Variables(4).ValuesAsNumpy()
daily_wind_speed_max = daily.Variables(5).ValuesAsNumpy()
daily_wind_gusts_max = daily.Variables(6).ValuesAsNumpy()
daily_wind_dir = daily.Variables(7).ValuesAsNumpy()

# Create your dataframe
daily_data = {
    "date": pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    ),
    "temp_max": daily_max,
    "temp_min": daily_min,
    "rh_max": daily_rh_max,
    "rh_mean": daily_rh_mean,
    "dew_point_mean": daily_dew_mean,
    "wind_speed_max": daily_wind_speed_max,
    "wind_gusts_max": daily_wind_gusts_max,
    "wind_dir": daily_wind_dir
}

# 1. Create the DataFrame
daily_dataframe = pd.DataFrame(data=daily_data)

daily_dataframe

Coordinates: 43.08087158203125°N -88.13955688476562°E
Elevation: 256.0 m asl
Timezone difference to GMT+0: 0s


,date,temp_max,temp_min,rh_max,rh_mean,dew_point_mean,wind_speed_max,wind_gusts_max,wind_dir
0,2025-10-13 00:00:00+00:00,61.355301,56.225300,99.0,83.083336,53.266636,10.853866,28.440001,137.966141
1,2025-10-14 00:00:00+00:00,62.705299,48.305302,96.0,82.000000,50.143677,19.228851,33.839996,23.498573
2,2025-10-15 00:00:00+00:00,58.205299,51.005302,99.0,85.625000,49.594967,16.135872,37.439999,52.659054
3,2025-10-16 00:00:00+00:00,63.785301,50.285301,89.0,76.833336,48.645164,14.291592,22.319998,93.030411
4,2025-10-17 00:00:00+00:00,73.055298,55.235302,77.0,67.583336,51.051937,21.829777,46.079998,178.345291
...,...,...,...,...,...,...,...,...,...
235,2026-06-05 00:00:00+00:00,83.765305,63.965302,98.0,70.291664,58.503410,16.700275,31.680000,201.463684
236,2026-06-06 00:00:00+00:00,83.855301,62.435303,92.0,77.500000,63.910748,15.778518,32.039997,237.166519
237,2026-06-07 00:00:00+00:00,80.615303,58.835304,91.0,68.333336,57.527550,16.099689,26.280001,76.118309
238,2026-06-08 00:00:00+00:00,77.915298,61.805302,88.0,72.458336,59.404339,17.873556,35.279999,98.444336
